# Train a Stable Diffusion 1.5 LoRA on a free Colab T4

Select **Runtime > Change runtime type > T4 GPU** before running.

This notebook clones the repository, installs the training extras with uv, and trains
from a processed dataset stored in Google Drive. Prepare the dataset locally with
`make prepare` first, or put 15 to 30 raw images in Drive and run the optional prep cell.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU")
print(torch.cuda.get_device_name(0))

## Clone and install

Replace `YOUR_GITHUB_USER` with your GitHub username or paste the full repository URL.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/YOUR_GITHUB_USER/lora-style-imagegen.git"
REPO_DIR = Path("/content/lora-style-imagegen")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!python -m pip install -q uv
!uv sync --extra train --no-dev

## Mount Google Drive

The default locations match this layout:

```text
MyDrive/lora-style-imagegen/
  data/raw/
  data/processed/
  outputs/lora-style/
```

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/lora-style-imagegen")
RAW_DIR = DRIVE_ROOT / "data/raw"
DATASET_DIR = DRIVE_ROOT / "data/processed"
OUTPUT_DIR = DRIVE_ROOT / "outputs/lora-style"
print(DRIVE_ROOT)

## Optional dataset preparation

Run this cell only if `data/processed/metadata.jsonl` does not exist yet and the raw
images are in Drive. You can skip it after preparing the dataset locally.

In [ ]:
if not (DATASET_DIR / "metadata.jsonl").exists():
    !uv run prepare-dataset --raw-dir {RAW_DIR} --output-dir {DATASET_DIR}

## Train

The default run uses rank 16, fp16, gradient checkpointing, 8-bit Adam, 800 optimizer
steps, and validation images every 200 steps. On a T4 this usually takes roughly 20 to
40 minutes for a small dataset. Outputs are written directly to Google Drive.

In [ ]:
import subprocess

subprocess.run(
    [
        "uv",
        "run",
        "train-lora",
        "--config",
        "configs/train_lora.yaml",
        "--dataset-dir",
        str(DATASET_DIR),
        "--output-dir",
        str(OUTPUT_DIR),
    ],
    check=True,
)

## Inspect the result

The safetensors adapter is at `outputs/lora-style/lora/pytorch_lora_weights.safetensors`.
Validation images are grouped by training step under `outputs/lora-style/validation/`.

In [ ]:
import json

from IPython.display import Image, display

metadata_path = OUTPUT_DIR / "lora/lora_metadata.json"
print(json.loads(metadata_path.read_text()))
for path in sorted((OUTPUT_DIR / "validation").rglob("*.png"))[-3:]:
    display(Image(filename=str(path), width=384))

## Optional TensorBoard

Loss and learning-rate events are stored under the output directory.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {OUTPUT_DIR}